# Opening Range Breakout 策略回测

使用 `vnpy_ctastrategy.backtesting.BacktestingEngine` 对开盘区间突破策略进行回测与参数优化。

## 工作流
1. 检查数据库中可用数据
2. 配置回测引擎 & 运行回测
3. 查看绩效统计 & 资金曲线
4. 参数优化（穷举 / 遗传算法）

> **注意**：回测数据从 vnpy SQLite 数据库加载。如数据不足，需先通过 DataManager 或 DataRecorder 导入/录制历史数据。

## 0. 环境准备

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

# 确保 strategies/ 目录在 sys.path 中（CWD 是 my_project/notebooks/）
project_dir = Path.cwd().parent  # my_project/
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from vnpy.trader.constant import Interval
from vnpy.trader.optimize import OptimizationSetting
from vnpy_ctastrategy.backtesting import BacktestingEngine
from strategies.opening_range_strategy import OpeningRangeBreakoutStrategy

import vnpy
print(f"vnpy {vnpy.__version__} loaded")

## 1. 检查数据库可用数据

In [ ]:
from vnpy_sqlite import Database

db = Database()
bar_overview = db.get_bar_overview()
bar_overview.sort(key=lambda x: x.count, reverse=True)

print(f"数据库中共 {len(bar_overview)} 个 Bar 数据集：\n")
print(f"{'合约':<18} {'周期':<6} {'起始时间':<22} {'结束时间':<22} {'数量':>6}")
print("-" * 80)
for b in bar_overview:
    vt = f"{b.symbol}.{b.exchange.value}"
    print(f"{vt:<18} {b.interval.value:<6} {str(b.start):<22} {str(b.end):<22} {b.count:>6}")

## 2. 配置回测参数

根据上面的数据可用性，选择合约和时间范围。

### 郑商所合约参数参考
| 品种 | 合约乘数 | 最小变动价位 | 手续费率（约） |
|------|---------|-------------|---------------|
| MA (甲醇) | 10吨/手 | 1 元/吨 | 2元/手 ≈ 0.8/万 |
| SA (纯碱) | 20吨/手 | 1 元/吨 | 3.5元/手 ≈ 1.2/万 |

In [ ]:
# ============ 回测配置 ============
# 修改以下参数适配你的数据

VT_SYMBOL = "MA605.CZCE"        # 合约代码
START = datetime(2026, 3, 31)    # 回测起始
END = datetime(2026, 3, 31)      # 回测结束

# 合约参数
SIZE = 10              # 合约乘数（MA=10吨/手）
PRICETICK = 1.0        # 最小变动价位
RATE = 0.8 / 10000     # 手续费率
SLIPPAGE = 1.0         # 滑点（1个最小变动价位）
CAPITAL = 100_000      # 初始资金

# 策略参数
STRATEGY_SETTING = {
    "opening_minutes": 5,
    "breakout_buffer_ticks": 1,
    "fixed_size": 1,
    "flat_after": "14:55",
    "no_new_entry_after": "14:40",
    "max_holding_minutes": 0,
    "trade_enabled": True,       # 回测中必须开启
}

## 3. 运行回测

In [ ]:
engine = BacktestingEngine()

engine.set_parameters(
    vt_symbol=VT_SYMBOL,
    interval=Interval.MINUTE,
    start=START,
    end=END,
    rate=RATE,
    slippage=SLIPPAGE,
    size=SIZE,
    pricetick=PRICETICK,
    capital=CAPITAL,
)

engine.add_strategy(OpeningRangeBreakoutStrategy, STRATEGY_SETTING)

engine.load_data()
engine.run_backtesting()
df = engine.calculate_result()
stats = engine.calculate_statistics()
print("\n回测完成")

## 4. 绩效可视化

In [ ]:
fig = engine.show_chart()
if fig:
    fig.show()

## 5. 成交明细

In [ ]:
trades = engine.get_all_trades()
if trades:
    print(f"共 {len(trades)} 笔成交：\n")
    print(f"{'时间':<22} {'方向':<6} {'开平':<10} {'价格':>10} {'数量':>6}")
    print("-" * 60)
    for t in trades:
        print(f"{str(t.datetime):<22} {t.direction.value:<6} {t.offset.value:<10} {t.price:>10.1f} {t.volume:>6.0f}")
else:
    print("无成交记录（数据不足或未触发信号）")

## 6. 参数优化（穷举）

使用 `OptimizationSetting` 定义参数搜索空间，目标为夏普比率。

In [ ]:
setting = OptimizationSetting()
setting.set_target("sharpe_ratio")
setting.add_parameter("opening_minutes", 3, 10, 1)
setting.add_parameter("breakout_buffer_ticks", 0, 3, 1)

# 穷举优化
result = engine.run_bf_optimization(setting)

if result:
    print(f"\n优化结果（Top 10）：\n")
    print(f"{'参数组合':<50} {'夏普比率':>10}")
    print("-" * 65)
    for params, target_value in result[:10]:
        print(f"{str(params):<50} {target_value:>10.4f}")
else:
    print("优化无结果（数据不足）")

## 7. 参数优化（遗传算法）

遗传算法适合参数空间较大的场景，使用 DEAP 库实现。

In [ ]:
# 遗传算法优化（参数空间同上）
# 需要安装 deap: pip install deap
try:
    ga_result = engine.run_ga_optimization(setting)
    if ga_result:
        print(f"\nGA 优化结果（Top 10）：\n")
        print(f"{'参数组合':<50} {'夏普比率':>10}")
        print("-" * 65)
        for params, target_value in ga_result[:10]:
            print(f"{str(params):<50} {target_value:>10.4f}")
    else:
        print("GA 优化无结果")
except ImportError:
    print("需要安装 deap 库: pip install deap")

## 8. 用最优参数重跑回测

In [ ]:
# 将上面优化得到的最优参数填入此处重新回测
BEST_SETTING = {
    "opening_minutes": 5,          # ← 替换为优化结果
    "breakout_buffer_ticks": 1,    # ← 替换为优化结果
    "fixed_size": 1,
    "flat_after": "14:55",
    "no_new_entry_after": "14:40",
    "max_holding_minutes": 0,
    "trade_enabled": True,
}

engine2 = BacktestingEngine()
engine2.set_parameters(
    vt_symbol=VT_SYMBOL,
    interval=Interval.MINUTE,
    start=START,
    end=END,
    rate=RATE,
    slippage=SLIPPAGE,
    size=SIZE,
    pricetick=PRICETICK,
    capital=CAPITAL,
)
engine2.add_strategy(OpeningRangeBreakoutStrategy, BEST_SETTING)
engine2.load_data()
engine2.run_backtesting()
df2 = engine2.calculate_result()
stats2 = engine2.calculate_statistics()

fig2 = engine2.show_chart()
if fig2:
    fig2.show()

---

## 下一步

1. **导入更多历史数据**：通过 VeighNa Trader GUI → 数据管理 (DataManager) 导入 CSV，或通过 DataRecorder 录制实时行情
2. **扩展回测时间范围**：积累至少 1-3 个月的 1 分钟数据后重新运行
3. **多合约回测**：对 SA、RM 等品种分别回测，观察策略普适性
4. **Walk-Forward 验证**：将数据分为 in-sample / out-of-sample 进行前推验证